# Labeled Set Manifest

Build a fetch-ready manifest for all 8,349 labeled stars by querying the MAST TIC catalog in bulk using Gaia DR3 IDs. The output is a single table recording, for each star, which missions have light curve data available (TESS, K2, Kepler) and the identifiers needed to fetch them.

This manifest is the single input to all downstream light curve fetching. No fetch logic lives here — only discovery.

## Steps
1. Bulk query TIC with all Gaia DR3 IDs in batches to retrieve TIC IDs
2. Query MAST observations for each TIC ID to check mission availability
3. Assign fetch status per star (`ready`, `no_tic`, `no_lightcurve`)
4. Save manifest to `data/processed/`

## Inputs
- `data/catalogs/gyronet_8k.csv`

## Outputs
- `data/processed/<to be decided>`

In [2]:
!pip install astroquery -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.1 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from astroquery.mast import Catalogs, Observations
import time

gyro = pd.read_csv('/content/drive/MyDrive/gyroformer/data/catalogs/gyronet_8k.csv')
print(f"Labeled set loaded: {gyro.shape}")

Labeled set loaded: (8349, 23)


In [4]:
# Query TIC in batches for all Gaia DR3 IDs
BATCH_SIZE = 1000

all_tic_results = []
dr3_ids = gyro['GaiaDR3_ID'].tolist()
batches = [dr3_ids[i:i+BATCH_SIZE] for i in range(0, len(dr3_ids), BATCH_SIZE)]

print(f"Total stars: {len(dr3_ids)}")
print(f"Batches: {len(batches)} x {BATCH_SIZE}")

for i, batch in enumerate(batches):
    try:
        result = Catalogs.query_criteria(catalog='TIC', GAIA=batch)
        all_tic_results.append(result.to_pandas()[['ID', 'GAIA', 'KIC']])
        print(f"Batch {i+1}/{len(batches)}: {len(result)} results")
    except Exception as e:
        print(f"Batch {i+1}/{len(batches)}: ERROR — {e}")
    time.sleep(1)

tic_df = pd.concat(all_tic_results, ignore_index=True)
tic_df.columns = ['TIC_ID', 'GaiaDR3_ID', 'KIC']
print(f"\nTotal TIC matches: {len(tic_df)}")

Total stars: 8349
Batches: 9 x 1000
Batch 1/9: 1001 results
Batch 2/9: 983 results
Batch 3/9: 973 results
Batch 4/9: 968 results
Batch 5/9: 967 results
Batch 6/9: 975 results
Batch 7/9: 988 results
Batch 8/9: 983 results
Batch 9/9: 347 results

Total TIC matches: 8185


In [5]:
print(f"Total rows: {len(tic_df)}")
print(f"Unique Gaia DR3 IDs: {tic_df['GaiaDR3_ID'].nunique()}")
print(f"Duplicated Gaia IDs: {tic_df['GaiaDR3_ID'].duplicated().sum()}")

# Look at a few duplicates
dupes = tic_df[tic_df['GaiaDR3_ID'].duplicated(keep=False)].sort_values('GaiaDR3_ID')
print(f"\nSample duplicates:")
print(dupes.head(10).to_string(index=False))

Total rows: 8185
Unique Gaia DR3 IDs: 8139
Duplicated Gaia IDs: 46

Sample duplicates:
    TIC_ID          GaiaDR3_ID     KIC
  17513640  144506309274714624    <NA>
 660230071  144506309274714624    <NA>
 660406774  149327461604162688    <NA>
 267953775  149327461604162688    <NA>
 270957488 2128108883031497856 9655402
1882698788 2128108883031497856    <NA>
1882702222 2128127265491233280    <NA>
 271165160 2128127265491233280 9595079
 271165997 2128142933532501376 9838036
1882705346 2128142933532501376    <NA>


In [6]:
def pick_best_tic(group):
    # Prefer row with KIC if available
    has_kic = group[group['KIC'].notna()]
    if len(has_kic) > 0:
        return has_kic.iloc[0]
    # Otherwise take lowest TIC ID
    return group.sort_values('TIC_ID').iloc[0]

tic_deduped = tic_df.groupby('GaiaDR3_ID').apply(pick_best_tic, include_groups=False).reset_index()
tic_deduped['GaiaDR3_ID'] = tic_deduped['GaiaDR3_ID'].astype(int)

print(f"After dedup: {len(tic_deduped)} unique Gaia DR3 IDs")
print(f"With KIC: {tic_deduped['KIC'].notna().sum()}")

After dedup: 8139 unique Gaia DR3 IDs
With KIC: 927


In [7]:
manifest = gyro[['GaiaDR3_ID', 'cluster', 'fiducial_age_Myr']].merge(
    tic_deduped, on='GaiaDR3_ID', how='left'
)

print(f"Total labeled stars: {len(manifest)}")
print(f"With TIC ID:    {manifest['TIC_ID'].notna().sum()}")
print(f"Without TIC ID: {manifest['TIC_ID'].isna().sum()}")

print(f"\nClusters with missing TIC entries:")
missing = manifest[manifest['TIC_ID'].isna()]
print(missing['cluster'].value_counts())

Total labeled stars: 8349
With TIC ID:    8139
Without TIC ID: 210

Clusters with missing TIC entries:
cluster
Praesepe        46
Pleiades        29
HPer            20
ScoCen          16
NGC2516         15
M50             12
APer            11
M34              9
M67              7
Hyades           5
M37              5
NGC2547          5
Collinder135     5
NGC3532          4
M35              4
NGC6811          4
IC2602           3
NGC2451A         2
NGC2281          2
IC2391           2
Ruprecht147      1
NGC2548          1
NGC1647          1
Taurus           1
Name: count, dtype: int64


In [8]:
# Query MAST observations for all stars with TIC IDs
tic_ids = manifest[manifest['TIC_ID'].notna()]['TIC_ID'].astype(int).tolist()

BATCH_SIZE = 500
batches = [tic_ids[i:i+BATCH_SIZE] for i in range(0, len(tic_ids), BATCH_SIZE)]

print(f"Stars with TIC ID: {len(tic_ids)}")
print(f"Batches: {len(batches)} x {BATCH_SIZE}")

obs_results = []
for i, batch in enumerate(batches):
    try:
        obs = Observations.query_criteria(
            target_name=batch,
            obs_collection=['TESS', 'K2', 'Kepler'],
            dataproduct_type='timeseries'
        )
        if len(obs) > 0:
            obs_df = obs['target_name', 'obs_collection'].to_pandas()
            obs_results.append(obs_df)
        print(f"Batch {i+1}/{len(batches)}: {len(obs)} results")
    except Exception as e:
        print(f"Batch {i+1}/{len(batches)}: ERROR — {e}")
    time.sleep(1)

obs_df = pd.concat(obs_results, ignore_index=True)
obs_df.columns = ['TIC_ID', 'mission']
obs_df['TIC_ID'] = obs_df['TIC_ID'].astype(str).str.strip().astype(int)
print(f"\nTotal observation records: {len(obs_df)}")
print(obs_df['mission'].value_counts())

Stars with TIC ID: 8139
Batches: 17 x 500
Batch 1/17: 193 results
Batch 2/17: 290 results
Batch 3/17: 166 results
Batch 4/17: 199 results
Batch 5/17: 201 results
Batch 6/17: 210 results
Batch 7/17: 731 results
Batch 8/17: 1426 results
Batch 9/17: 1278 results
Batch 10/17: 262 results
Batch 11/17: 1080 results
Batch 12/17: 228 results
Batch 13/17: 786 results
Batch 14/17: 1157 results
Batch 15/17: 683 results
Batch 16/17: 256 results
Batch 17/17: 52 results

Total observation records: 9198
mission
TESS    9198
Name: count, dtype: int64


In [9]:
# NGC6811 star we know has Kepler data — KIC 9838496 from earlier
obs_test = Observations.query_criteria(
    target_name=['9838496'],
    obs_collection=['Kepler'],
    dataproduct_type='timeseries'
)
print(f"Kepler query by KIC: {len(obs_test)} results")
print(obs_test['target_name', 'obs_collection', 'obs_id'][:3] if len(obs_test) > 0 else "No results")

# Also try with ktwo prefix for K2
obs_test2 = Observations.query_criteria(
    target_name=['246868747'],
    obs_collection=['K2'],
    dataproduct_type='timeseries'
)
print(f"\nK2 query by EPIC: {len(obs_test2)} results")
print(obs_test2['target_name', 'obs_collection', 'obs_id'][:3] if len(obs_test2) > 0 else "No results")

Kepler query by KIC: 0 results
No results

K2 query by EPIC: 0 results
No results


In [10]:
from astroquery.mast import Observations

# NGC6811 star — KIC 9838496, we have its Gaia DR3 ID from earlier
# Get its coordinates from TIC first
test_tic = Catalogs.query_criteria(catalog='TIC', KIC=9838496)
print(test_tic['ID', 'GAIA', 'KIC', 'ra', 'dec'])

# Then query MAST by position
ra = float(test_tic['ra'][0])
dec = float(test_tic['dec'][0])
print(f"\nQuerying MAST at RA={ra:.4f}, Dec={dec:.4f}")

obs_pos = Observations.query_region(
    f"{ra} {dec}",
    radius="2 arcsec"
)
obs_pos_df = obs_pos['target_name', 'obs_collection', 'dataproduct_type'].to_pandas()
print(obs_pos_df[obs_pos_df['obs_collection'].isin(['Kepler','K2','TESS'])])

    ID            GAIA          KIC          ra              dec       
--------- ------------------- ------- ---------------- ----------------
271253897 2128514774619770368 9838496 294.842889612016 46.6913923325056

Querying MAST at RA=294.8429, Dec=46.6914
      target_name obs_collection dataproduct_type
0        TESS FFI           TESS            image
1        TESS FFI           TESS            image
2        TESS FFI           TESS            image
3        TESS FFI           TESS            image
4        TESS FFI           TESS            image
5        TESS FFI           TESS            image
6        TESS FFI           TESS            image
7        TESS FFI           TESS            image
8        TESS FFI           TESS            image
9        TESS FFI           TESS            image
71  kplr009838496         Kepler       timeseries
72  kplr009838487         Kepler       timeseries


In [11]:
# Format KIC IDs as kplr target names and query in bulk
kic_stars = manifest[manifest['KIC'].notna()].copy()
kic_stars['kplr_id'] = kic_stars['KIC'].astype(int).apply(lambda x: f"kplr{x:09d}")

print(f"Stars with KIC IDs: {len(kic_stars)}")
print(f"Sample kplr IDs: {kic_stars['kplr_id'].head(5).tolist()}")

# Test bulk query
test_kplr = kic_stars['kplr_id'].head(50).tolist()
obs_kepler = Observations.query_criteria(
    target_name=test_kplr,
    obs_collection=['Kepler'],
    dataproduct_type='timeseries'
)
print(f"\nKepler results for 50 test stars: {len(obs_kepler)}")
if len(obs_kepler) > 0:
    print(obs_kepler['target_name', 'obs_collection'][:5])

Stars with KIC IDs: 927
Sample kplr IDs: ['kplr008264442', 'kplr008264565', 'kplr008330056', 'kplr008330169', 'kplr004937546']

Kepler results for 50 test stars: 63
 target_name  obs_collection
------------- --------------
kplr004848304         Kepler
kplr004936404         Kepler
kplr004936404         Kepler
kplr004936597         Kepler
kplr004936672         Kepler


In [13]:
# Full Kepler query for all KIC stars
kplr_ids = kic_stars['kplr_id'].tolist()
BATCH_SIZE = 500
batches = [kplr_ids[i:i+BATCH_SIZE] for i in range(0, len(kplr_ids), BATCH_SIZE)]

kepler_results = []
for i, batch in enumerate(batches):
    try:
        obs = Observations.query_criteria(
            target_name=batch,
            obs_collection=['Kepler'],
            dataproduct_type='timeseries'
        )
        if len(obs) > 0:
            df = obs.to_pandas()[['target_name']]
            kepler_results.append(df)
        print(f"Batch {i+1}/{len(batches)}: {len(obs)} results")
    except Exception as e:
        print(f"Batch {i+1}/{len(batches)}: ERROR — {e}")
    time.sleep(1)

if kepler_results:
    kepler_df = pd.concat(kepler_results, ignore_index=True)
    kepler_df['target_name'] = kepler_df['target_name'].str.strip()
    kepler_has_data = set(kepler_df['target_name'].unique())
    print(f"\nUnique KIC stars with Kepler data: {len(kepler_has_data)}")
else:
    print("No Kepler results found")

Batch 1/2: 358 results
Batch 2/2: 261 results

Unique KIC stars with Kepler data: 526


In [14]:
# Check K2 target name format from our earlier NGC1817 result
# Format is ktwo246868747 — these are EPIC IDs
# TIC doesn't store EPIC IDs but we can query K2 by TIC ID via a different approach
# First let's just test — do any of our TIC IDs return K2 results?

test_tics = manifest[manifest['TIC_ID'].notna()]['TIC_ID'].astype(int).head(500).tolist()

obs_k2_test = Observations.query_criteria(
    target_name=test_tics,
    obs_collection=['K2'],
    dataproduct_type='timeseries'
)
print(f"K2 results querying by TIC ID: {len(obs_k2_test)}")
if len(obs_k2_test) > 0:
    print(obs_k2_test.to_pandas()[['target_name', 'obs_collection']].head())

K2 results querying by TIC ID: 0


In [15]:
# K2 campaigns and their target clusters — known from literature
# Query MAST for K2 timeseries in the footprint of a known K2 cluster
# Test with Pleiades (Campaign 4, center ~56.75, +24.11)

obs_pleiades = Observations.query_criteria(
    obs_collection=['K2'],
    dataproduct_type='timeseries',
    s_ra=[50.0, 63.0],
    s_dec=[18.0, 30.0]
)
print(f"K2 observations in Pleiades region: {len(obs_pleiades)}")
if len(obs_pleiades) > 0:
    print(obs_pleiades.to_pandas()[['target_name', 'obs_collection', 's_ra', 's_dec']].head())

K2 observations in Pleiades region: 18810
      target_name obs_collection       s_ra      s_dec
0  polar210680671             K2  52.002645  18.091164
1  polar210681653             K2  55.934940  18.104357
2  polar210682270             K2  53.315640  18.113134
3  polar210683034             K2  59.898240  18.123952
4  polar210680070             K2  58.126560  18.083030


In [20]:
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np

# Get Pleiades stars with coordinates from TIC
pleiades_manifest = manifest[manifest['cluster'] == 'Pleiades'].copy()
pleiades_tic = Catalogs.query_criteria(
    catalog='TIC',
    GAIA=pleiades_manifest['GaiaDR3_ID'].tolist()
).to_pandas()[['ID', 'GAIA', 'ra', 'dec']]
pleiades_tic.columns = ['TIC_ID', 'GaiaDR3_ID', 'ra', 'dec']
pleiades_tic['TIC_ID'] = pleiades_tic['TIC_ID'].astype(str)
pleiades_tic['GaiaDR3_ID'] = pleiades_tic['GaiaDR3_ID'].astype(int)

# Merge coords onto manifest
pleiades_valid = pleiades_manifest.merge(pleiades_tic[['GaiaDR3_ID', 'ra', 'dec']], on='GaiaDR3_ID', how='left')
pleiades_valid = pleiades_valid.dropna(subset=['ra', 'dec']).copy()
print(f"Pleiades stars with valid coords: {len(pleiades_valid)}")

# Get K2 observations in Pleiades region
obs_pleiades = Observations.query_criteria(
    obs_collection=['K2'],
    dataproduct_type='timeseries',
    s_ra=[50.0, 63.0],
    s_dec=[18.0, 30.0]
)
obs_pleiades_df = obs_pleiades.to_pandas()[['target_name', 's_ra', 's_dec']]
obs_pleiades_df['EPIC'] = obs_pleiades_df['target_name'].str.extract(r'(\d+)').astype(int)

# Crossmatch by position
our_coords = SkyCoord(ra=pleiades_valid['ra'].values*u.deg, dec=pleiades_valid['dec'].values*u.deg)
k2_coords = SkyCoord(ra=obs_pleiades_df['s_ra'].values*u.deg, dec=obs_pleiades_df['s_dec'].values*u.deg)
idx, sep, _ = our_coords.match_to_catalog_sky(k2_coords)
matched = sep < 2*u.arcsec

# Assign EPIC IDs
epic_values = np.full(len(pleiades_valid), np.nan)
epic_values[matched] = obs_pleiades_df.iloc[idx[matched]]['EPIC'].values
pleiades_valid['EPIC'] = epic_values

print(f"Pleiades stars matched to K2: {matched.sum()} / {len(pleiades_valid)}")
print(pleiades_valid[pleiades_valid['EPIC'].notna()][['GaiaDR3_ID', 'TIC_ID', 'EPIC']].head())

Pleiades stars with valid coords: 830
Pleiades stars matched to K2: 737 / 830
          GaiaDR3_ID     TIC_ID         EPIC
3  51609980593783424  440702109  210866482.0
4  50101210121892992   14112924  210769047.0
5  53105728725582976   15818292  210940618.0
6  53332571717238400   14225108  210967582.0
7  53341642688604416   14285398  210948982.0


In [21]:
# Clusters that need K2 coordinate crossmatch
# These are clusters not covered by TESS-only or Kepler
k2_needed = manifest[~manifest['cluster'].isin(['NGC6811', 'NGC6819', 'NGC6866'])]['cluster'].unique()

# Get cluster centers from our labeled stars' TIC coordinates
# First we need coords for all stars — let's get them from gyro catalog directly
# TARS has ra/dec, use that where available, else we'll need TIC

# Check if gyro has ra/dec
print(gyro.columns.tolist())

['GaiaDR3_ID', 'cluster', 'subgroup', 'fiducial_age_Myr', 'Prot', 'BPRP_0', 'G_0', 'e_BPRP_0', 'dr3_parallax_zpt_corrected', 'dr3_ruwe', 'mem_prob_hdbscan', 'cmd_exclude', 'dered_exclude', 'phot_exclude', 'phot_bp_rp_excess_factor', 'astrometric_excess_noise_sig', 'phot_bp_rp_excess_factor_t', 'astrometric_excess_noise_sig_t', 'noise_sig_detected', 'spec_coverage', 'e_Prot_true', 'e_Prot_imputed', 'mem_prob_source']


In [23]:
tars = pd.read_csv('/content/drive/MyDrive/gyroformer/data/catalogs/tars_table_2.csv',
                   usecols=['dr3_source_id', 'ra', 'dec'])
print(f"TARS loaded: {tars.shape}")

TARS loaded: (944056, 3)


In [24]:
# Build a coordinate table for all labeled stars
# Source 1: TARS (has ra/dec for the 2823 matched stars)
tars_coords = tars[tars['dr3_source_id'].isin(gyro['GaiaDR3_ID'])][['dr3_source_id', 'ra', 'dec']].copy()
tars_coords.columns = ['GaiaDR3_ID', 'ra', 'dec']
print(f"Coords from TARS: {len(tars_coords)}")

# Source 2: TIC (for remaining stars)
remaining_ids = gyro[~gyro['GaiaDR3_ID'].isin(tars_coords['GaiaDR3_ID'])]['GaiaDR3_ID'].tolist()
print(f"Still need coords for: {len(remaining_ids)} stars — fetching from TIC...")

tic_coord_results = []
BATCH_SIZE = 1000
batches = [remaining_ids[i:i+BATCH_SIZE] for i in range(0, len(remaining_ids), BATCH_SIZE)]

for i, batch in enumerate(batches):
    try:
        result = Catalogs.query_criteria(catalog='TIC', GAIA=batch)
        df = result.to_pandas()[['GAIA', 'ra', 'dec']]
        df.columns = ['GaiaDR3_ID', 'ra', 'dec']
        tic_coord_results.append(df)
        print(f"Batch {i+1}/{len(batches)}: {len(df)} results")
    except Exception as e:
        print(f"Batch {i+1}/{len(batches)}: ERROR — {e}")
    time.sleep(1)

tic_coords = pd.concat(tic_coord_results, ignore_index=True)
tic_coords['GaiaDR3_ID'] = tic_coords['GaiaDR3_ID'].astype(int)

# Combine both sources
all_coords = pd.concat([tars_coords, tic_coords], ignore_index=True).drop_duplicates(subset='GaiaDR3_ID')
print(f"\nTotal stars with coordinates: {len(all_coords)} / {len(gyro)}")

Coords from TARS: 2823
Still need coords for: 5526 stars — fetching from TIC...
Batch 1/6: 1001 results
Batch 2/6: 975 results
Batch 3/6: 962 results
Batch 4/6: 959 results
Batch 5/6: 985 results
Batch 6/6: 522 results

Total stars with coordinates: 8200 / 8349


In [25]:
# Compute cluster bounding boxes with padding
coords_with_cluster = gyro[['GaiaDR3_ID', 'cluster']].merge(all_coords, on='GaiaDR3_ID', how='inner')

cluster_boxes = coords_with_cluster.groupby('cluster').agg(
    ra_min=('ra', 'min'),
    ra_max=('ra', 'max'),
    dec_min=('dec', 'min'),
    dec_max=('dec', 'max'),
    n_stars=('GaiaDR3_ID', 'count')
).reset_index()

# Add 0.5 deg padding on each side
PAD = 0.5
cluster_boxes['ra_min'] -= PAD
cluster_boxes['ra_max'] += PAD
cluster_boxes['dec_min'] -= PAD
cluster_boxes['dec_max'] += PAD

print(cluster_boxes[['cluster', 'ra_min', 'ra_max', 'dec_min', 'dec_max', 'n_stars']].to_string(index=False))

     cluster     ra_min     ra_max    dec_min    dec_max  n_stars
        APer  35.143862  65.761720  40.545736  57.204282      238
Collinder135 104.753747 113.962750 -40.731883 -33.349332      246
        HPer  33.981573  35.509358  56.488093  57.778615      400
      Hyades  35.911783  80.784617   3.136396  38.042457      177
      IC2391 126.689973 133.786613 -55.636821 -50.358713      113
      IC2602 154.502054 166.749779 -67.523322 -60.143074      193
         M34  39.678279  41.383171  41.961122  43.537846       75
         M35  91.130515  93.144516  23.419754  25.146600      533
         M37  87.326415  88.820611  31.843243  33.259256      343
         M50 104.896557 106.508516  -9.187090  -7.585357      645
         M67 131.713465 133.895244  10.689996  12.817831      276
     NGC1647  70.393877  72.629672  18.474119  20.024317       34
     NGC1750  74.474241  76.905924  22.185418  24.673357       39
     NGC1758  75.489804  76.790557  23.227322  24.405531       21
     NGC18

In [27]:
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np

# Skip pure Kepler clusters
kepler_only = ['NGC6811', 'NGC6819', 'NGC6866']
k2_clusters_to_query = cluster_boxes[~cluster_boxes['cluster'].isin(kepler_only)].copy()

all_k2_matches = []

for _, row in k2_clusters_to_query.iterrows():
    cluster = row['cluster']
    try:
        obs = Observations.query_criteria(
            obs_collection=['K2'],
            dataproduct_type='timeseries',
            s_ra=[row['ra_min'], row['ra_max']],
            s_dec=[row['dec_min'], row['dec_max']]
        )
        if len(obs) == 0:
            print(f"{cluster:15s}: 0 K2 observations in region")
            continue

        obs_df = obs.to_pandas()[['target_name', 's_ra', 's_dec']]
        obs_df['EPIC'] = obs_df['target_name'].str.extract(r'(\d+)').astype(float)

        # Get our stars in this cluster with coords
        our_stars = coords_with_cluster[coords_with_cluster['cluster'] == cluster].dropna(subset=['ra','dec'])
        if len(our_stars) == 0:
            print(f"{cluster:15s}: no coord data for our stars")
            continue

        our_coords = SkyCoord(ra=our_stars['ra'].values*u.deg, dec=our_stars['dec'].values*u.deg)
        k2_coords = SkyCoord(ra=obs_df['s_ra'].values*u.deg, dec=obs_df['s_dec'].values*u.deg)

        idx, sep, _ = our_coords.match_to_catalog_sky(k2_coords)
        matched = sep < 2*u.arcsec

        epic_values = np.full(len(our_stars), np.nan)
        epic_values[matched] = obs_df.iloc[idx[matched]]['EPIC'].values

        result = our_stars[['GaiaDR3_ID', 'cluster']].copy()
        result['EPIC'] = epic_values
        result['has_k2'] = matched
        all_k2_matches.append(result)

        print(f"{cluster:15s}: {matched.sum():4d} / {len(our_stars):4d} matched to K2")

    except Exception as e:
        print(f"{cluster:15s}: ERROR — {e}")
    time.sleep(0.5)

k2_match_df = pd.concat(all_k2_matches, ignore_index=True)
print(f"\nTotal K2 matches: {k2_match_df['has_k2'].sum()} / {len(k2_match_df)}")

APer           : 0 K2 observations in region
Collinder135   : 0 K2 observations in region
HPer           : 0 K2 observations in region
Hyades         :  124 /  177 matched to K2


IC2391         : 0 K2 observations in region
IC2602         : 0 K2 observations in region
M34            : 0 K2 observations in region
M35            :    5 /  533 matched to K2


M37            : 0 K2 observations in region
M50            : 0 K2 observations in region
M67            :  120 /  276 matched to K2
NGC1647        :   34 /   34 matched to K2
NGC1750        :   39 /   39 matched to K2
NGC1758        :   21 /   21 matched to K2
NGC1817        :   55 /   55 matched to K2


NGC2281        : 0 K2 observations in region
NGC2451A       : 0 K2 observations in region
NGC2516        : 0 K2 observations in region
NGC2547        : 0 K2 observations in region
NGC2548        : 0 K2 observations in region
NGC3532        : 0 K2 observations in region
NGC752         : 0 K2 observations in region
Pleiades       :  747 /  839 matched to K2
Praesepe       :  960 /  961 matched to K2
Ruprecht147    :   36 /   41 matched to K2
ScoCen         : 1033 / 1033 matched to K2
Taurus         :   92 /   92 matched to K2

Total K2 matches: 3266 / 4101


In [28]:
print(type(obs_results[0]))
print(obs_results[0].columns.tolist())
print(obs_results[0].head(2))

<class 'pandas.core.frame.DataFrame'>
['target_name', 'obs_collection']
  target_name obs_collection
0   271041303           TESS
1   270698296           TESS


In [29]:
# TESS availability
tess_df = pd.concat(obs_results, ignore_index=True)
tess_df['TIC_ID'] = tess_df['target_name'].astype(int)
tess_has_data = set(tess_df['TIC_ID'].unique())

# Kepler availability — from kepler_has_data set (kplr formatted)
# Map back to KIC integers
kepler_kic_has_data = set(int(k.replace('kplr', '')) for k in kepler_has_data)

# K2 EPIC IDs
k2_epic_map = k2_match_df[k2_match_df['has_k2']][['GaiaDR3_ID', 'EPIC']].copy()
k2_epic_map['EPIC'] = k2_epic_map['EPIC'].astype(int)

# Assemble
manifest_final = gyro[['GaiaDR3_ID', 'cluster', 'fiducial_age_Myr']].copy()
manifest_final = manifest_final.merge(tic_deduped[['GaiaDR3_ID', 'TIC_ID', 'KIC']], on='GaiaDR3_ID', how='left')
manifest_final = manifest_final.merge(k2_epic_map, on='GaiaDR3_ID', how='left')

manifest_final['TIC_ID'] = pd.to_numeric(manifest_final['TIC_ID'], errors='coerce')
manifest_final['has_tess'] = manifest_final['TIC_ID'].isin(tess_has_data)
manifest_final['has_kepler'] = manifest_final['KIC'].isin(kepler_kic_has_data)
manifest_final['has_k2'] = manifest_final['EPIC'].notna()

# Fetch status
def fetch_status(row):
    if pd.isna(row['TIC_ID']) and not row['has_kepler'] and not row['has_k2']:
        return 'no_tic'
    if not row['has_tess'] and not row['has_kepler'] and not row['has_k2']:
        return 'no_lightcurve'
    return 'ready'

manifest_final['fetch_status'] = manifest_final.apply(fetch_status, axis=1)

print("=== Manifest summary ===")
print(f"Total stars:      {len(manifest_final)}")
print(f"\nfetch_status:")
print(manifest_final['fetch_status'].value_counts())
print(f"\nhas_tess:   {manifest_final['has_tess'].sum()}")
print(f"has_kepler: {manifest_final['has_kepler'].sum()}")
print(f"has_k2:     {manifest_final['has_k2'].sum()}")

=== Manifest summary ===
Total stars:      8349

fetch_status:
fetch_status
ready            4447
no_lightcurve    3726
no_tic            176
Name: count, dtype: int64

has_tess:   2998
has_kepler: 526
has_k2:     3266


In [30]:
print(manifest_final.dtypes)
print(f"\nDuplicate GaiaDR3_IDs: {manifest_final['GaiaDR3_ID'].duplicated().sum()}")
print(f"\nSample ready stars:")
print(manifest_final[manifest_final['fetch_status']=='ready'].head(3).to_string(index=False))
print(f"\nSample no_lightcurve stars:")
print(manifest_final[manifest_final['fetch_status']=='no_lightcurve'].head(3).to_string(index=False))

GaiaDR3_ID            int64
cluster              object
fiducial_age_Myr    float64
TIC_ID              float64
KIC                  object
EPIC                float64
has_tess               bool
has_kepler             bool
has_k2                 bool
fetch_status         object
dtype: object

Duplicate GaiaDR3_IDs: 0

Sample ready stars:
        GaiaDR3_ID cluster  fiducial_age_Myr      TIC_ID  KIC        EPIC  has_tess  has_kepler  has_k2 fetch_status
145203704587705088  Taurus              2.01  61262664.0 <NA> 247584146.0     False       False    True        ready
145217379763796992  Taurus              2.01 118555907.0 <NA> 247600777.0      True       False    True        ready
146675954953119104  Taurus              1.57 238408266.0 <NA> 247609913.0      True       False    True        ready

Sample no_lightcurve stars:
         GaiaDR3_ID cluster  fiducial_age_Myr      TIC_ID  KIC  EPIC  has_tess  has_kepler  has_k2  fetch_status
5290646440127895936 NGC2516             263.0 372

In [31]:
manifest_final['TIC_ID'] = manifest_final['TIC_ID'].astype('Int64')
manifest_final['EPIC'] = manifest_final['EPIC'].astype('Int64')
manifest_final['KIC'] = pd.to_numeric(manifest_final['KIC'], errors='coerce').astype('Int64')

print(manifest_final.dtypes)
print(manifest_final.head(3).to_string(index=False))

GaiaDR3_ID            int64
cluster              object
fiducial_age_Myr    float64
TIC_ID                Int64
KIC                   Int64
EPIC                  Int64
has_tess               bool
has_kepler             bool
has_k2                 bool
fetch_status         object
dtype: object
         GaiaDR3_ID cluster  fiducial_age_Myr    TIC_ID  KIC  EPIC  has_tess  has_kepler  has_k2  fetch_status
5290646440127895936 NGC2516             263.0 372912811 <NA>  <NA>     False       False   False no_lightcurve
5290651323511794816 NGC2516             263.0 364398861 <NA>  <NA>     False       False   False no_lightcurve
5290652663541576832 NGC2516             263.0 364398907 <NA>  <NA>     False       False   False no_lightcurve


In [32]:
output_path = '/content/drive/MyDrive/gyroformer/data/processed/labeled_set_lightcurve_availability.csv'
manifest_final.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Shape: {manifest_final.shape}")

Saved to /content/drive/MyDrive/gyroformer/data/processed/labeled_set_lightcurve_availability.csv
Shape: (8349, 10)
